# CREACIÓN DE UN MODELO USANDO TRANSFORMERS DE HUGGING FACE

# PASO 1 - INSTALAR TRANSFORMERS Y DATASETS DE HUGGING FACE

In [1]:
!pip install transformers datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 16.0 MB/s eta 0:00:00


# PASO 2 - IMPORTAR LIBRERIAS

In [23]:
import torch
from torch.utils.data import DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from torch.utils.data import Dataset

# PASO 3 - CARGAMOS EL DATASET SMS_SPAM_COLLECTION

In [24]:
# Cargar el dataset
dataset = load_dataset("sms_spam")

# Ver las primeras filas del dataset para confirmar el nombre de las columnas
print(dataset['train'][0])

# Tokenizador BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Preprocesamiento de datos
def preprocess_function(examples):
    # Ajusta la columna de acuerdo con el nombre correcto
    return tokenizer(examples['sms'], padding="max_length", truncation=True, max_length=512)  # Si la columna es 'message'

# Aplicar preprocesamiento a los datos
encoded_dataset = dataset.map(preprocess_function, batched=True)

README.md:   0%|          | 0.00/4.98k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/359k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5574 [00:00<?, ? examples/s]

{'sms': 'Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...\n', 'label': 0}


Map:   0%|          | 0/5574 [00:00<?, ? examples/s]

# PASO 4 - PREPROCESAR LOS DATOS Y DIVIDIMOS DATASET EN TRAIN Y TEST

In [25]:
dataset

DatasetDict({
    train: Dataset({
        features: ['sms', 'label'],
        num_rows: 5574
    })
})

In [26]:
# Dividir en train y test
train_texts, test_texts, train_labels, test_labels = train_test_split(
    encoded_dataset['train']['sms'],
    encoded_dataset['train']['label'],
    test_size=0.1
)

class EmailDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(self.texts[idx], truncation=True, padding='max_length', max_length=512, return_tensors='pt')
        item = {key: encoding[key].squeeze() for key in encoding}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# Crear los datasets
train_dataset = EmailDataset(train_texts, train_labels, tokenizer)
test_dataset = EmailDataset(test_texts, test_labels, tokenizer)

# PASO 5 - CREAR EL MODELO EN BASE A BERT PARA LA CLASIFICIÓN

In [28]:
# Cargar el modelo BERT preentrenado para clasificación
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# PASO 6 -  CONFIGURAR EL MODELO

In [29]:
# Configurar el optimizador
optimizer = AdamW(model.parameters(), lr=2e-5)

# Entrenamiento
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8)

# PASO 7 - ENTRENAMOS EL MODELO

In [ ]:
for epoch in range(3):  # Entrenamos por 3 épocas
    model.train()
    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()

        optimizer.step()

    print(f"Epoch {epoch + 1}: Loss {loss.item()}")

# PASO 8 - EVALUAMOS EL MODELO

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print(f"Test Accuracy: {accuracy * 100:.2f}%")

{'eval_loss': 0.054129261523485184, 'eval_runtime': 7.8002, 'eval_samples_per_second': 142.946, 'eval_steps_per_second': 17.948, 'epoch': 3.0}


# PASO 9 - PROBAMOS EL CLASIFICADOR

In [ ]:
# Función para predecir si un email es spam o no
def predict_email(email_text):
    inputs = tokenizer(email_text, return_tensors='pt', truncation=True, padding=True, max_length=128).to(device)
    outputs = model(**inputs)
    prediction = torch.argmax(outputs.logits, dim=1)
    return "spam" if prediction.item() == 1 else "ham"  # 'ham' es no spam

# Probar con un correo electrónico
email_text = "Free money! Claim your reward now!"
prediction = predict_email(email_text)
print(f"El correo es: {prediction}")


El correo es: spam
